In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("12-string-functions")
dfs = register_views(spark)
emp = dfs["employees"]

# ── Section 1: concat and concat_ws ──────────────────────────────────────────
# concat(col1, lit(" "), col2)     → NULL if ANY argument is NULL.
# concat_ws(sep, col1, col2, ...)  → skips NULL arguments; never returns NULL
#                                    unless ALL inputs are NULL.
print("=== Section 1: concat vs concat_ws ===")
emp.withColumn("full_name",    F.concat(F.col("first_name"), F.lit(" "), F.col("last_name"))) \
   .withColumn("full_name_ws", F.concat_ws(" ", "first_name", "last_name")) \
   .select("emp_id", "full_name", "full_name_ws").show(5)

# ── Section 2: upper, lower, initcap, trim, ltrim, rtrim ─────────────────────
# initcap capitalises the first letter of each word (useful for email display).
# trim/ltrim/rtrim remove whitespace; handy when source data has padding.
print("=== Section 2: case functions ===")
emp.withColumn("upper_name",  F.upper("first_name")) \
   .withColumn("lower_name",  F.lower("last_name")) \
   .withColumn("title_email", F.initcap("email")) \
   .select("emp_id", "upper_name", "lower_name", "title_email").show(3)

print("=== Section 2: trim / ltrim with artificially padded strings ===")
spark.createDataFrame([("  Emma  ",), ("  Frank ",)], ["name"]) \
     .withColumn("trimmed",  F.trim("name")) \
     .withColumn("ltrimmed", F.ltrim("name")) \
     .show()

# ── Section 3: length, substring ─────────────────────────────────────────────
# length(col)             → character count (not byte count for multibyte).
# substring(col, pos, len)→ 1-based position; returns "" if pos > length.
print("=== Section 3: length and substring ===")
emp.withColumn("fname_len", F.length("first_name")) \
   .withColumn("fname_2",   F.substring("first_name", 1, 2)) \
   .select("emp_id", "first_name", "fname_len", "fname_2").show(5)

# ── Section 4: split and array element access ─────────────────────────────────
# split(col, pattern) → ArrayType(StringType()); access elements with [index].
# NULL input → split returns NULL → element access also returns NULL.
print("=== Section 4: split email into user/domain parts ===")
emp.withColumn("email_parts",  F.split("email", "@")) \
   .withColumn("email_user",   F.split("email", "@")[0]) \
   .withColumn("email_domain", F.split("email", "@")[1]) \
   .select("emp_id", "email", "email_user", "email_domain").show(5)
# Note: emp 22 (NULL email) → all three derived columns are NULL

# ── Section 5: regexp_replace and regexp_extract ─────────────────────────────
# regexp_replace(col, pattern, replacement) → global replace (all occurrences).
# regexp_extract(col, pattern, groupIndex)  → returns empty string (not NULL)
#   when there is no match; groupIndex 0 = whole match, 1 = first group.
print("=== Section 5: regexp_replace – strip hyphens from phone ===")
emp.withColumn("clean_phone", F.regexp_replace("phone", "-", "")) \
   .select("emp_id", "phone", "clean_phone").show(3)

print("=== Section 5: regexp_extract – area code (first 3 digits) ===")
emp.withColumn("area_code", F.regexp_extract("phone", r"^(\d{3})", 1)) \
   .select("emp_id", "phone", "area_code").show(3)

# ── Section 6: like, rlike, contains, startswith, endswith ───────────────────
# like     → SQL LIKE (% wildcard, _ single char); case-sensitive by default.
# rlike    → full Java regex; use (?i) flag for case-insensitive matching.
# startswith / endswith → Column methods; used in .filter() or .withColumn().
print("=== Section 6: like – last names starting with C ===")
emp.filter(F.col("last_name").like("C%")).select("emp_id", "last_name").show()

print("=== Section 6: like – corp.com email addresses ===")
emp.filter(F.col("email").like("%@corp.com")).select("emp_id", "email").show(3)

print("=== Section 6: rlike – job_title contains 'engineer' (case-insensitive) ===")
emp.filter(F.col("job_title").rlike(r"(?i)engineer")).select("emp_id", "job_title").show()

print("=== Section 6: startswith – first names beginning with J ===")
emp.filter(F.col("first_name").startswith("J")).select("emp_id", "first_name").show()

# ── Section 7: instr and locate ──────────────────────────────────────────────
# instr(col, substr)           → 1-based index of first match; 0 if not found.
# locate(substr, col[, pos])   → same semantics; argument order is reversed!
# Both return 0 (not NULL) when the substring is absent.
print("=== Section 7: instr – position of '@' in email ===")
emp.withColumn("at_pos", F.instr("email", "@")) \
   .select("emp_id", "email", "at_pos").show(3)

print("=== Section 7: locate – searching for absent substring returns 0 ===")
emp.withColumn("not_found", F.locate("xyz", "email")).select("emp_id", "not_found").show(3)

# ── Section 8: lpad and rpad ──────────────────────────────────────────────────
# lpad(col, length, pad_char) → left-pad to fixed width; common for ID formatting.
# rpad pads on the right side.
print("=== Section 8: lpad – zero-padded emp_id ===")
emp.withColumn("padded_id", F.lpad(F.col("emp_id").cast("string"), 4, "0")) \
   .select("emp_id", "padded_id").show(3)

# ── Section 9: translate ──────────────────────────────────────────────────────
# translate(col, matching_chars, replace_chars) → character-level substitution.
# Each char in matching_chars maps to the corresponding char in replace_chars.
# If replace_chars is shorter, unmatched chars in matching_chars are deleted.
print("=== Section 9: translate – remove vowels from first_name ===")
emp.withColumn("no_vowels", F.translate("first_name", "aeiouAEIOU", "")) \
   .select("emp_id", "first_name", "no_vowels").show(3)

# ── Section 10: format_string (printf-style) ──────────────────────────────────
# format_string(format, col1, col2, …) → Java String.format / printf semantics.
# %04d = zero-pad integer to 4 digits; %-20s = left-align in 20 chars; %,.0f = thousand-sep float.
# NULL salary is coalesced to 0.0 to avoid NULL propagation into the format string.
print("=== Section 10: format_string – display card per employee ===")
emp.withColumn("display",
    F.format_string(
        "ID: %04d | %-20s | $%,.0f",
        F.col("emp_id"),
        F.col("first_name"),
        F.coalesce(F.col("salary"), F.lit(0.0))
    )
).select("display").show(5, truncate=False)

# ── Section 11: NULL handling in string functions ─────────────────────────────
# concat:    returns NULL if ANY input is NULL.
# concat_ws: skips NULL inputs; only returns NULL if ALL inputs are NULL.
# Demonstrate with emp 22 who has NULL email.
print("=== Section 11: NULL behavior – concat vs concat_ws (emp 22, NULL email) ===")
emp.withColumn("concat_result",    F.concat(F.col("email"), F.lit(" <active>"))) \
   .withColumn("concat_ws_result", F.concat_ws(" <active> ", "email")) \
   .select("emp_id", "email", "concat_result", "concat_ws_result") \
   .filter(F.col("emp_id") == 22).show()
# concat_result → NULL (because email is NULL)
# concat_ws_result → " <active> " (concat_ws skips the NULL email)

# ── Section 12: soundex and levenshtein (fuzzy matching) ─────────────────────
# soundex(col)                  → 4-char phonetic code; matches words that sound alike.
# levenshtein(col1, col2)       → edit distance (insertions + deletions + substitutions).
# Useful for de-duplication, fuzzy name matching, and data quality checks.
from pyspark.sql.functions import levenshtein

print("=== Section 12: soundex – phonetic code for last_name ===")
emp.withColumn("soundex_last", F.soundex("last_name")) \
   .select("emp_id", "last_name", "soundex_last").show(5)

print("=== Section 12: levenshtein – near-duplicate first names (distance 1-2) ===")
emp.crossJoin(emp.select(F.col("first_name").alias("other_name"))) \
   .withColumn("dist", F.levenshtein("first_name", "other_name")) \
   .filter((F.col("dist") > 0) & (F.col("dist") <= 2)) \
   .select("first_name", "other_name", "dist").show(5)
# crossJoin is expensive on large data; restrict to small lookup tables or use blocking keys first

stop_and_wait(spark)